# 디렉토리에서 문서를 로드하는 방법 (2026년 최신 권장 사용법)

이번 튜토리얼은 다음의 내용을 포함합니다.
- 와일드카드 패턴을 포함하여 파일 시스템에서 로드하는 방법
- 파일 I/O 에 멀티스레딩을 사용하는 방법
- 특정 파일 유형(예: 코드)을 파싱하기 위해 사용자 정의 로더 클래스를 사용하는 방법
- 디코딩 오류와 같은 오류를 처리하는 방법

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `DirectoryLoader(path, glob, loader_cls, show_progress, use_multithreading, silent_errors)` | `pathlib.Path.glob()` + `ThreadPoolExecutor` + `tqdm` 로 같은 기능의 작은 로더 직접 작성 |
| 기본 `loader_cls` = `UnstructuredFileLoader` | **`langchain-unstructured`** 의 `UnstructuredLoader` (여러 파일 경로 리스트도 직접 받음) |
| `TextLoader` | `pathlib` 기반 `TextFileLoader` (08 노트북과 동일) |
| `PythonLoader` | `tokenize.open()` 으로 소스 인코딩을 감지해 읽는 `PythonFileLoader` |

In [ ]:
# 설치
# !pip install -qU langchain-core tqdm charset-normalizer
# !pip install -qU langchain-unstructured "unstructured[md]"   # UnstructuredLoader 사용 시

## 기본 파일 로더 준비

`DirectoryLoader` 는 "파일 경로 → 로더" 를 만들어 주는 **팩토리**(`loader_cls`)를 받습니다. 여기서도 같은 방식으로, 파일 경로를 받는 로더 클래스를 넘기도록 설계합니다.

In [ ]:
import logging
import tokenize
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Callable, Iterator

from charset_normalizer import from_path
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

logger = logging.getLogger(__name__)


class TextFileLoader(BaseLoader):
    """텍스트 파일 1개 → Document 1개 (구 TextLoader)"""

    def __init__(self, file_path: str | Path, encoding: str = "utf-8", autodetect_encoding: bool = True):
        self.file_path = Path(file_path)
        self.encoding = encoding
        self.autodetect_encoding = autodetect_encoding

    def lazy_load(self) -> Iterator[Document]:
        try:
            text = self.file_path.read_text(encoding=self.encoding)
        except UnicodeDecodeError:
            if not self.autodetect_encoding:
                raise
            best = from_path(self.file_path).best()
            if best is None:
                raise
            text = str(best)
        yield Document(page_content=text, metadata={"source": str(self.file_path)})


class PythonFileLoader(BaseLoader):
    """파이썬 소스 파일 로더 (구 PythonLoader): PEP 263 인코딩 선언을 존중"""

    def __init__(self, file_path: str | Path):
        self.file_path = Path(file_path)

    def lazy_load(self) -> Iterator[Document]:
        with tokenize.open(self.file_path) as f:  # 소스 파일 인코딩 자동 감지
            yield Document(page_content=f.read(), metadata={"source": str(self.file_path)})

## DirectoryLoader 대체 구현

- `glob` 매개변수로 로드할 파일 형식을 제어합니다. (`**` 는 하위 폴더 재귀)
- `show_progress=True` 이면 `tqdm` 진행 표시줄을 보여줍니다.
- `use_multithreading=True` 이면 여러 스레드로 파일을 동시에 읽습니다. (결과 순서는 유지)
- `silent_errors=True` 이면 실패한 파일은 경고만 남기고 건너뜁니다.

In [ ]:
class FolderLoader(BaseLoader):
    """디렉토리의 파일들을 지정한 로더로 읽어오는 로더 (구 DirectoryLoader)"""

    def __init__(
        self,
        path: str | Path,
        glob: str = "**/*",
        loader_cls: Callable[[Path], BaseLoader] = TextFileLoader,
        *,
        exclude: tuple[str, ...] = (".ipynb_checkpoints", ".venv", "node_modules"),
        show_progress: bool = False,
        use_multithreading: bool = False,
        max_workers: int = 4,
        silent_errors: bool = False,
    ) -> None:
        self.path = Path(path)
        self.glob = glob
        self.loader_cls = loader_cls
        self.exclude = exclude
        self.show_progress = show_progress
        self.use_multithreading = use_multithreading
        self.max_workers = max_workers
        self.silent_errors = silent_errors

    def _files(self) -> list[Path]:
        return sorted(
            p
            for p in self.path.glob(self.glob)
            if p.is_file() and not any(part in self.exclude for part in p.parts)
        )

    def _load_file(self, file: Path) -> list[Document]:
        try:
            return self.loader_cls(file).load()
        except Exception as e:
            if not self.silent_errors:
                raise
            logger.warning("건너뜀: %s (%s)", file, e)
            return []

    def lazy_load(self) -> Iterator[Document]:
        files = self._files()
        if self.use_multithreading:
            with ThreadPoolExecutor(max_workers=self.max_workers) as pool:
                results = pool.map(self._load_file, files)  # 입력 순서 유지
                yield from self._progress(results, len(files))
        else:
            yield from self._progress(map(self._load_file, files), len(files))

    def _progress(self, results, total) -> Iterator[Document]:
        if self.show_progress:
            from tqdm.auto import tqdm

            results = tqdm(results, total=total)
        for docs in results:
            yield from docs

In [ ]:
# 디렉토리 로더 초기화 (상위 폴더의 모든 .md 파일)
loader = FolderLoader("../", glob="**/*.md")
# 문서 로드
docs = loader.load()
# 문서 개수 계산
len(docs)

In [ ]:
# 페이지 내용 출력
print(docs[0].page_content[:100])

기본적으로 진행 상태 표시줄은 표시되지 않습니다. `show_progress=True` 옵션으로 진행상황을 확인할 수 있습니다.

**참고**
- 진행 상태 표시줄을 표시하려면 `tqdm` 라이브러리를 설치(예: `pip install tqdm`)
- `show_progress` 매개변수를 `True` 로 설정

In [ ]:
loader = FolderLoader("../", glob="**/*.md", show_progress=True)  # 디렉토리 로더 설정
docs = loader.load()  # 문서 로드

기본적으로 로딩은 하나의 스레드에서 이루어집니다.

여러 스레드를 활용하려면 `use_multithreading=True` 로 설정하세요. (파일 I/O 대기 시간이 긴 경우 효과적)

In [ ]:
loader = FolderLoader("../", glob="**/*.md", use_multithreading=True)  # 디렉토리 로더 설정
docs = loader.load()  # 문서 로드

## loader_cls 변경

`loader_cls` 에 파일 경로를 받는 로더 클래스(또는 함수)를 지정하면 파일 형식에 맞는 파서를 쓸 수 있습니다.

### UnstructuredLoader 사용 (책의 기본값 `UnstructuredFileLoader` 대체)

`UnstructuredLoader` 는 Markdown 의 제목·목록 등을 **요소** 단위로 파싱합니다. 반면 `TextFileLoader` 는 파일 전체를 그대로 읽습니다.

In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = FolderLoader("../", glob="**/*.md", loader_cls=UnstructuredLoader)
docs = loader.load()

print(len(docs))  # 요소 단위이므로 파일 수보다 많습니다.
print(docs[0].metadata.get("category"), "|", docs[0].page_content[:100])

> 참고: `UnstructuredLoader` 는 **파일 경로 리스트**를 직접 받을 수도 있습니다.
>
> ```python
> files = sorted(Path("../").glob("**/*.md"))
> docs = UnstructuredLoader(file_path=[str(f) for f in files]).load()
> ```

### TextFileLoader 사용

In [ ]:
# loader_cls 를 TextFileLoader 로 지정합니다. (Markdown 문법이 그대로 남습니다)
loader = FolderLoader("../", glob="**/*.md", loader_cls=TextFileLoader)

# 문서 로드
docs = loader.load()

In [ ]:
# 문서 페이지 내용 출력
print(docs[0].page_content[:100])

### 파이썬 소스 코드

Python 소스 코드 파일을 로드해야 하는 경우, 위에서 만든 `PythonFileLoader` 를 사용합니다.

> 코드를 RAG 에 넣을 때는 `langchain_text_splitters.RecursiveCharacterTextSplitter.from_language(Language.PYTHON, ...)` 로 함수/클래스 경계를 고려해 분할하면 좋습니다.

In [ ]:
# 현재폴더(.) 의 .py 파일을 모두 조회하여 PythonFileLoader 로 로드
loader = FolderLoader(".", glob="**/*.py", loader_cls=PythonFileLoader)

In [ ]:
# 문서 로드
docs = loader.load()
docs

In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=500, chunk_overlap=0
)
code_chunks = python_splitter.split_documents(docs)
len(code_chunks)

### 확장자별로 다른 로더 쓰기

`loader_cls` 에는 함수도 넘길 수 있으므로, 확장자에 따라 로더를 고르는 팩토리를 만들 수 있습니다.

In [ ]:
def auto_loader(path: Path) -> BaseLoader:
    if path.suffix == ".py":
        return PythonFileLoader(path)
    if path.suffix in {".txt", ".md"}:
        return TextFileLoader(path)
    return UnstructuredLoader(str(path))  # 그 외(pdf, docx, pptx ...)는 Unstructured 로


loader = FolderLoader("./data", glob="**/*", loader_cls=auto_loader, silent_errors=True, show_progress=True)
mixed_docs = loader.load()
len(mixed_docs)